# Feature Extraction
Tahap ini bertujuan untuk mengubah teks bersih menjadi representasi numerik untuk model BiGRU dan IndoBERT. Kita akan melakukan:
1. Split Dataset (Train, Val, Test)
2. Word-level Tokenization (untuk BiGRU)
3. Subword Tokenization (untuk IndoBERT)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle

df = pd.read_csv('../Dataset/dataset_clean_final.csv')
print(f"Total data: {len(df)}")

Total data: 70379


## 1. Split Dataset
Kita akan membagi data menjadi 70% Train, 15% Validation, dan 15% Test.

In [2]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['label'])

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

Train size: 49265
Val size: 10557
Test size: 10557


## 2. Word-level Tokenization (BiGRU)
Karena kendala library, kita akan menggunakan tokenizer kustom yang kompatibel dengan NumPy/PyTorch/TensorFlow.

In [3]:
from collections import Counter

class JudolTokenizer:
    def __init__(self, num_words=20000, oov_token='<OOV>'):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def fit_on_texts(self, texts):
        all_words = [word for text in texts for word in text.split()]
        counts = Counter(all_words)
        most_common = counts.most_common(self.num_words - 1)
        
        self.word_index = {self.oov_token: 1}
        for i, (word, _) in enumerate(most_common, 2):
            self.word_index[word] = i
        
        self.index_word = {i: word for word, i in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            seq = [self.word_index.get(word, self.word_index[self.oov_token]) for word in text.split()]
            sequences.append(seq)
        return sequences

def pad_sequences_manual(sequences, maxlen=100):
    padded = np.zeros((len(sequences), maxlen), dtype=int)
    for i, seq in enumerate(sequences):
        if len(seq) > 0:
            trunc = seq[:maxlen]
            padded[i, :len(trunc)] = trunc
    return padded

MAX_WORDS = 20000
MAX_LENGTH = 100

tokenizer = JudolTokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(train_df['text'])

def prepare_bigru(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences_manual(sequences, maxlen=MAX_LENGTH)
    return padded

X_train_bigru = prepare_bigru(train_df['text'])
X_val_bigru = prepare_bigru(val_df['text'])
X_test_bigru = prepare_bigru(test_df['text'])

print(f"BiGRU Input Shape: {X_train_bigru.shape}")

BiGRU Input Shape: (49265, 100)


## 3. Subword Tokenization (IndoBERT)
Menggunakan tokenizer dari Hugging Face.

In [4]:
from transformers import AutoTokenizer

model_name = "indobenchmark/indobert-base-p2"
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

def prepare_indobert(texts):
    return bert_tokenizer(
        list(texts),
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='np'
    )

train_encodings = prepare_indobert(train_df['text'])
val_encodings = prepare_indobert(val_df['text'])
test_encodings = prepare_indobert(test_df['text'])

print(f"IndoBERT Input ID Shape: {train_encodings['input_ids'].shape}")

: 

## 4. Simpan Hasil
Simpan tokenizer dan data yang sudah di-encode untuk tahap training.

In [ ]:
# Simpan BiGRU Tokenizer
with open('../Dataset/tokenizer_bigru.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Simpan Data Tensor (Opsional, atau bisa langsung lanjut ke training notebook)
print("Feature extraction selesai dan tokenizer disimpan.")